
TkinterGUI for Student Performance Prediction
- Handles the "only one class" error by ensuring the dataset has two classes before training.
- If loaded/generated data contains a single class, it re-computes Result using a score proxy and median threshold.
- Uses stratify in train_test_split only when there are >=2 classes.
- Run as a standalone script or in an environment that supports tkinter.


In [3]:
import os
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg


# Helper functions 

def generate_synthetic(n=200, seed=42):
    np.random.seed(seed)
    study_hours = np.random.randint(1, 11, n)
    attendance = np.random.randint(40, 101, n)
    previous_score = np.random.randint(30, 96, n)
    assignment_marks = np.random.randint(30, 101, n)

    # Create a normalized final score proxy in range ~0-100
    final_score = 0.3 * previous_score + 0.3 * assignment_marks + 0.4 * (attendance)
    # Use median threshold to ensure both classes exist (balanced-ish)
    median_threshold = np.median(final_score)
    result = ["Pass" if s >= median_threshold else "Fail" for s in final_score]

    df = pd.DataFrame({
        "StudyHours": study_hours,
        "Attendance": attendance,
        "PreviousScore": previous_score,
        "AssignmentMarks": assignment_marks,
        "FinalScoreProxy": final_score,
        "Result": result
    })
    return df

def load_csv(path):
    df = pd.read_csv(path)
    required = {'StudyHours','Attendance','PreviousScore','AssignmentMarks','Result'}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"CSV must contain columns: {required}")
    # If FinalScoreProxy not present, it's fine; we'll compute if needed
    return df

def ensure_two_classes(df):
    """
    Ensure df['Result'] contains at least two classes (0 and 1).
    If only one class exists, compute a score proxy and reassign Result using median threshold.
    Returns modified df and a flag indicating whether modification occurred.
    """
    df = df.copy()
    # Map textual labels to numeric if needed
    if df['Result'].dtype == object:
        df['Result'] = df['Result'].map({'Pass':1, 'Fail':0})
    # Fill missing numeric features
    df[['StudyHours','Attendance','PreviousScore','AssignmentMarks']] = df[['StudyHours','Attendance','PreviousScore','AssignmentMarks']].fillna(df.median())
    # If Result has only one unique value, recompute using a score proxy
    if df['Result'].nunique() < 2:
        # Compute a final score proxy (same formula used in generation)
        final_score = 0.3 * df['PreviousScore'] + 0.3 * df['AssignmentMarks'] + 0.4 * df['Attendance']
        median_threshold = final_score.median()
        df['Result'] = (final_score >= median_threshold).astype(int)
        df['FinalScoreProxy'] = final_score
        modified = True
    else:
        modified = False
    # Ensure integer type
    df['Result'] = df['Result'].astype(int)
    return df, modified

def classification_metrics(y_true, y_pred):
    return {
        'Accuracy': round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1-Score': round(f1_score(y_true, y_pred, zero_division=0), 3)
    }


In [4]:
# Core ML Pipeline
def train_and_evaluate(df):
    df_checked, modified = ensure_two_classes(df)
    X = df_checked[['StudyHours','Attendance','PreviousScore','AssignmentMarks']]
    y = df_checked['Result']

    # Use stratify only if there are at least 2 classes
    strat = y if y.nunique() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=strat)

    # Logistic Regression
    log_model = LogisticRegression(max_iter=1000)
    log_model.fit(X_train, y_train)
    log_pred = log_model.predict(X_test)

    # Linear Regression -> thresholded classification (use PreviousScore as proxy)
    lin_model = LinearRegression()
    lin_model.fit(X_train[['StudyHours','Attendance','AssignmentMarks']], X_train['PreviousScore'])
    lin_pred_scores = lin_model.predict(X_test[['StudyHours','Attendance','AssignmentMarks']])
    lin_pred = (lin_pred_scores >= 50).astype(int)

    # Decision Tree
    dt_model = DecisionTreeClassifier(random_state=42)
    dt_model.fit(X_train, y_train)
    dt_pred = dt_model.predict(X_test)

    # Random Forest
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    # Metrics
    results = {
        'Logistic Regression': classification_metrics(y_test, log_pred),
        'Linear Regression (thresholded)': classification_metrics(y_test, lin_pred),
        'Decision Tree': classification_metrics(y_test, dt_pred),
        'Random Forest': classification_metrics(y_test, rf_pred)
    }

    metrics_df = pd.DataFrame(results).T.reset_index().rename(columns={'index':'Model'})

    # Determine best model by F1-Score
    metrics_df[['Accuracy','Precision','Recall','F1-Score']] = metrics_df[['Accuracy','Precision','Recall','F1-Score']].astype(float)
    best_row = metrics_df.loc[metrics_df['F1-Score'].idxmax()]
    best_model_name = best_row['Model']
    best_model_f1 = best_row['F1-Score']

    pred_map = {
        'Logistic Regression': log_pred,
        'Linear Regression (thresholded)': lin_pred,
        'Decision Tree': dt_pred,
        'Random Forest': rf_pred
    }
    best_pred = pred_map[best_model_name]
    cm = confusion_matrix(y_test, best_pred)

    return {
        'metrics_df': metrics_df.round(3),
        'best_model_name': best_model_name,
        'best_model_f1': round(best_model_f1, 3),
        'confusion_matrix': cm,
        'y_test': y_test.reset_index(drop=True),
        'predictions': {
            'Logistic Regression': pd.Series(log_pred),
            'Linear Regression (thresholded)': pd.Series(lin_pred),
            'Decision Tree': pd.Series(dt_pred),
            'Random Forest': pd.Series(rf_pred)
        },
        'modified_labels': modified
    }

In [5]:
# Tkinter GUI 

class StudentMLApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Student Performance Prediction - GUI (Fixed)")
        self.root.geometry("980x640")
        self.df = None
        self.results = None

        ctrl_frame = ttk.Frame(root, padding=8)
        ctrl_frame.pack(side=tk.TOP, fill=tk.X)

        ttk.Button(ctrl_frame, text="Generate Synthetic Data", command=self.generate_data).pack(side=tk.LEFT, padx=6)
        ttk.Button(ctrl_frame, text="Load CSV", command=self.load_file).pack(side=tk.LEFT, padx=6)
        ttk.Button(ctrl_frame, text="Train & Evaluate", command=self.train_models).pack(side=tk.LEFT, padx=6)
        ttk.Button(ctrl_frame, text="Show Pass/Fail Pie", command=self.show_pie).pack(side=tk.LEFT, padx=6)
        ttk.Button(ctrl_frame, text="Show Actual vs Pred Points", command=self.show_points).pack(side=tk.LEFT, padx=6)
        ttk.Button(ctrl_frame, text="Exit", command=root.quit).pack(side=tk.RIGHT, padx=6)

        mid_frame = ttk.Frame(root, padding=8)
        mid_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        left = ttk.LabelFrame(mid_frame, text="Dataset Preview", padding=6)
        left.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=6, pady=6)

        self.tree = ttk.Treeview(left, columns=("StudyHours","Attendance","PreviousScore","AssignmentMarks","Result"), show='headings', height=12)
        for col in ("StudyHours","Attendance","PreviousScore","AssignmentMarks","Result"):
            self.tree.heading(col, text=col)
            self.tree.column(col, width=100, anchor='center')
        self.tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        vsb = ttk.Scrollbar(left, orient="vertical", command=self.tree.yview)
        vsb.pack(side=tk.RIGHT, fill='y')
        self.tree.configure(yscrollcommand=vsb.set)

        right = ttk.LabelFrame(mid_frame, text="Model Comparison (Metrics)", padding=6)
        right.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=6, pady=6)

        self.metrics_tree = ttk.Treeview(right, columns=("Model","Accuracy","Precision","Recall","F1-Score"), show='headings', height=12)
        for col in ("Model","Accuracy","Precision","Recall","F1-Score"):
            self.metrics_tree.heading(col, text=col)
            self.metrics_tree.column(col, width=120, anchor='center')
        self.metrics_tree.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        vsb2 = ttk.Scrollbar(right, orient="vertical", command=self.metrics_tree.yview)
        vsb2.pack(side=tk.RIGHT, fill='y')
        self.metrics_tree.configure(yscrollcommand=vsb2.set)

        bottom = ttk.LabelFrame(root, text="Plots", padding=6)
        bottom.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True, padx=6, pady=6)
        self.fig = plt.Figure(figsize=(9,3.5), dpi=100)
        self.canvas = FigureCanvasTkAgg(self.fig, master=bottom)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def generate_data(self):
        self.df = generate_synthetic(n=200)
        self._populate_dataset_preview()
        messagebox.showinfo("Data Generated", "Synthetic dataset generated (200 rows).")

    def load_file(self):
        path = filedialog.askopenfilename(filetypes=[("CSV files","*.csv"),("All files","*.*")])
        if not path:
            return
        try:
            self.df = load_csv(path)
            self._populate_dataset_preview()
            messagebox.showinfo("Loaded", f"Loaded dataset from:\n{os.path.basename(path)}")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to load CSV:\n{e}")

    def _populate_dataset_preview(self):
        for i in self.tree.get_children():
            self.tree.delete(i)
        if self.df is None:
            return
        preview = self.df.head(50)
        for _, row in preview.iterrows():
            vals = (row['StudyHours'], row['Attendance'], row['PreviousScore'], row['AssignmentMarks'], row['Result'])
            self.tree.insert('', tk.END, values=vals)

    def train_models(self):
        if self.df is None:
            messagebox.showwarning("No Data", "Please load a CSV or generate synthetic data first.")
            return
        try:
            self.results = train_and_evaluate(self.df)
        except Exception as e:
            messagebox.showerror("Training Error", f"An error occurred during training:\n{e}")
            return

        for i in self.metrics_tree.get_children():
            self.metrics_tree.delete(i)
        for _, row in self.results['metrics_df'].iterrows():
            self.metrics_tree.insert('', tk.END, values=(row['Model'], row['Accuracy'], row['Precision'], row['Recall'], row['F1-Score']))

        best = self.results['best_model_name']
        f1 = self.results['best_model_f1']
        modified = self.results.get('modified_labels', False)
        info_msg = f"Best model by F1-Score:\n{best} (F1 = {f1})"
        if modified:
            info_msg += "\n\nNote: The dataset contained a single class and labels were recomputed using a score proxy (median threshold) to allow training."
        messagebox.showinfo("Training Complete", info_msg)

        self._plot_confusion_matrix(self.results['confusion_matrix'], title=f"Confusion Matrix ({best})")

    def show_pie(self):
        if self.df is None:
            messagebox.showwarning("No Data", "Load or generate data first.")
            return
        df_checked, _ = ensure_two_classes(self.df)
        counts = df_checked['Result'].value_counts().sort_index()
        labels = ['Fail (0)','Pass (1)']
        sizes = [counts.get(0,0), counts.get(1,0)]
        self.fig.clf()
        ax = self.fig.add_subplot(111)
        colors = ['salmon','lightgreen']
        ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, explode=(0.05,0.05))
        ax.set_title("Pass vs Fail Distribution")
        self.canvas.draw()

    def show_points(self):
        if self.results is None:
            messagebox.showwarning("No Results", "Train models first to view Actual vs Predicted points.")
            return
        y_test = self.results['y_test']
        preds = self.results['predictions']
        self.fig.clf()
        ax = self.fig.add_subplot(111)
        idx = np.arange(len(y_test))
        ax.scatter(idx, y_test, label='Actual', marker='o', color='black', s=50)
        jitter = 0.08
        colors = {'Logistic Regression':'tab:blue','Linear Regression (thresholded)':'tab:orange','Decision Tree':'tab:green','Random Forest':'tab:red'}
        markers = {'Logistic Regression':'x','Linear Regression (thresholded)':'^','Decision Tree':'s','Random Forest':'D'}
        offsets = {'Logistic Regression':-3*jitter,'Linear Regression (thresholded)':-jitter,'Decision Tree':jitter,'Random Forest':3*jitter}
        for name, series in preds.items():
            ax.scatter(idx + offsets[name], series.values, label=name, marker=markers[name], color=colors[name], s=40, alpha=0.9)
        ax.set_yticks([0,1])
        ax.set_yticklabels(['Fail (0)','Pass (1)'])
        ax.set_xlabel("Test Sample Index")
        ax.set_title("Actual vs Predicted (Points) — Test Set")
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        self.canvas.draw()

    def _plot_confusion_matrix(self, cm, title="Confusion Matrix"):
        self.fig.clf()
        ax = self.fig.add_subplot(111)
        im = ax.imshow(cm, cmap='Blues')
        ax.set_xticks([0,1])
        ax.set_yticks([0,1])
        ax.set_xticklabels(['Pred_Fail','Pred_Pass'])
        ax.set_yticklabels(['Actual_Fail','Actual_Pass'])
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i,j], ha='center', va='center', color='black', fontsize=12)
        ax.set_title(title)
        self.canvas.draw()

In [9]:
# Run the app
if __name__ == "__main__":
    root = tk.Tk()
    app = StudentMLApp(root)
    root.mainloop()